# NSCG + O*NET Data Integration

## Summary

**74,016 respondents across 123 NSCG occupation categories; 50 columns, one row per person.** The earlier 6,849-person subset used narrow occupation matches. The revised rules allow shared broader profiles and reviewed occupation-group means.

Two **left joins** keep NSCG respondents on the left: `N3OCPR` links to the reviewed category rules, then the screened `ONET_PROFILE_ID` links to an occupation profile. Both use `validate='many_to_one'` and preserve all **79,800 eligible respondents**. The CSV contains only the 74,016 included respondents (92.8%).

Use actual O*NET `.00` records only; ignore `.01`, `.02` and other subtypes. Average each of the six Work Values equally across the same fully rated members. Missing ratings are not zero. Several NSCG categories may share a profile; a category with several occupations uses a group mean.

| Selection | People |
|---|---:|
| Included: full rating coverage | 39,084 |
| Included: partial rating coverage | 34,932 |
| Excluded: no Work Values | 5,068 |
| Excluded: uncertain mapping | 548 |
| Excluded: supervisor screen | 168 |

Mapping assumptions are flagged separately: 23,249 included respondents use a proxy; this overlaps with full/partial coverage. Single-member proxies remain flagged. Full coverage does not imply an exact match to each person's job.

The CSV retains nine personal importance items, six occupational Work Values, original satisfaction codes and survey weights. No overall fit score or prediction model is computed.

[Integrated CSV](nscg2023_onet30_2_subset.csv)

## Why the matching rules changed

The original selection was too restrictive for NSCG's broad occupation categories. The main limits were title wording and occupation relationships, not capitalization.

| Issue | Original rule | Revised rule |
|---|---|---|
| Capitalization and punctuation | Already handled by `casefold()` and title normalization. | No change; these were not the reason for the small subset. |
| Different wording | Selected titles had to match after normalization. | Use reviewed occupation scope and codes; titles can differ. This is not an automatic fuzzy-name match. |
| O*NET specialty codes | Each selected SOC had to have only one official O*NET candidate. | Use an actual `.00` profile when available; ignore `.01`, `.02` and other subtypes. Do not create a `.00` rating by removing a suffix. |
| One NSCG category to several O*NET occupations | Broad groups were outside the single-occupation selection. | Define the member set, then average each Work Value equally over the same fully rated occupations. |
| Several NSCG categories to one O*NET occupation | Selected SOC and O*NET codes had to be unique across NSCG categories. | Allow shared broader profiles while keeping the original NSCG categories separate. |

**One-to-many example:** `732510` (preschool and kindergarten teachers) uses the mean of `25-2011.00` and `25-2012.00`.

**Many-to-one example:** `632530` (secondary computer/math/science teachers) and `632540` (secondary social-science teachers) both use `25-2031.00`.

These relationships describe occupation categories, not people with multiple jobs. Build one profile per included NSCG category first, then attach it with validated many-to-one left joins. This keeps one row per respondent.

The expansion also allows partial rating coverage and flagged proxies. Categories with no rated profile or uncertain mapping remain excluded. More coverage does not mean every person's occupation is exactly matched.


### Data preparation

Tested with Python 3.12, numpy 2.2.1, pandas 2.2.3, openpyxl 3.1.5 and IPython 9.9.0:

```bash
python -m pip install numpy==2.2.1 pandas==2.2.3 openpyxl==3.1.5 ipython==9.9.0 ipykernel==7.1.0
```

1. Download the [NSCG 2023 public-use ZIP](https://ncses.nsf.gov/822/assets/0/files/college_grads_2023.zip), September 2025 release. Extract into `data/nscg/`; the ZIP already contains `pcg23Public/`.
2. Save [Table A-2](https://ncses.nsf.gov/pubs/nsf25322/assets/technical-tables/tables/nsf25322-taba-002.xlsx) as `data/nscg/nsf25322-taba-002.xlsx`.
3. Run all cells from the repository or notebook directory. O*NET30.2 files are in the repository. The reviewed member lists are embedded below; private review files are not required.

### Sources

NSCG 2023 data, dictionary, SAS labels and Table A-2; O*NET30.2 Work Values, occupation definitions, content model and scales; official SOC2018/O*NET-SOC2019 crosswalk.

The [2C study crosswalk](https://github.com/Break-Through-Tech/Swytch-2C-predicting-job-satisfaction-from-person-job-fit/blob/af0514f4286457966dca81bfffbd08b2c43f59e0/data/crosswalks/ncses_to_soc2018.csv) helped enumerate candidates. Final memberships are study-defined, not an official NSCG crosswalk. [SOC definitions](https://www.bls.gov/soc/2018/major_groups.htm) support the scope review.


## 1. Load source fields

Keep the original 24 NSCG fields. `OBSNUM` identifies each respondent; `N3OCPR` is the occupation key. Exclude job-aspect satisfaction (`SAT*`). Public-file completeness may include NSCG imputations.


In [1]:
from pathlib import Path
import hashlib
import json
import platform
import re
import warnings
import numpy as np
import pandas as pd
from IPython.display import display, HTML

NOTEBOOK_REL=Path('notebooks/possible data integration/nscg_onet_data_integration.ipynb')
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/NOTEBOOK_REL).is_file()),None)
if ROOT is None:
    raise FileNotFoundError('Run from this repository or the notebook directory.')
NSCG=ROOT/'data/nscg/pcg23Public'
ONET=ROOT/'data/onet'
OUT=ROOT/'data/private/nscg_onet_work_values'
SUBSET_CSV=ROOT/'notebooks/possible data integration/nscg2023_onet30_2_subset.csv'
SOURCE={
    'survey':NSCG/'epcg23.csv',
    'dictionary':NSCG/'Dpcg23.xlsx',
    'labels':NSCG/'Ppcg23.sas',
    'hierarchy':ROOT/'data/nscg/nsf25322-taba-002.xlsx',
    'crosswalk':ONET/'onetsoc2019_to_soc2018_crosswalk.xlsx',
    'values':ONET/'db_30_2_text/Work Values.txt',
    'occupations':ONET/'db_30_2_text/Occupation Data.txt',
    'content':ONET/'db_30_2_text/Content Model Reference.txt',
    'scales':ONET/'db_30_2_text/Scales Reference.txt',
}
missing=[str(p.relative_to(ROOT)) for p in SOURCE.values() if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing local inputs:\n- '+'\n- '.join(missing)+
                            '\nSee Data preparation above for downloads and paths.')
warnings.filterwarnings('ignore',message='Workbook contains no default style')
pd.set_option('display.max_colwidth',95)
pd.set_option('display.max_rows',35)

In [2]:
dictionary=pd.read_excel(SOURCE['dictionary'])
display(dictionary.head())
definitions=dictionary.set_index('SAS_NAME')
assert definitions.index.is_unique
formats={};current=None
for line in SOURCE['labels'].read_text(encoding='cp1252').splitlines():
    start=re.match(r'\s*VALUE\s+\$?(\S+)',line)
    if start:
        current=start.group(1);formats[current]={};continue
    value=re.match(r'\s*"?([^"=]*?)"?\s*=\s*"(.*)"',line)
    if current and value:
        formats[current][value.group(1).strip()]=value.group(2).split(': ',1)[-1]
    if line.strip()==';':current=None
def labels_for(variable):
    name=definitions.at[variable,'SAS_FMT'].lstrip('$').rstrip('.')
    return formats[name]
def code_for(variable,label):
    found=[code for code,text in labels_for(variable).items() if text==label]
    assert len(found)==1,(variable,label,found)
    return found[0]
fac=dictionary.loc[dictionary.SAS_NAME.str.startswith('FAC'),'SAS_NAME'].tolist()
core=['OBSNUM','REFYR','AGE','SEX_2023','LFSTAT','JOBSATIS','N3OCPR','N3OCPRX',
      'N2OCPRMG','N3OCPRNG','OCEDRLP','DGRDG','WTSURVY','SUPWK','WAPRI']+fac
raw=pd.read_csv(SOURCE['survey'],usecols=core,dtype='string',keep_default_na=False)
display(raw.head())
survey=raw[core].apply(lambda col:col.str.strip())
assert survey.OBSNUM.is_unique and survey.OBSNUM.ne('').all()
assert survey.REFYR.eq('2023').all()
assert len(fac)==9
display(pd.DataFrame(labels_for('JOBSATIS').items(),columns=['JOBSATIS code','Label']))
display(pd.DataFrame(labels_for('FACIND').items(),columns=['FAC code','Label']))

,SAS_NAME,SESTAT_NAME,DESCRIPTION,SOURCE,QUESTION,DOMAIN,SAS_FMT
0,ACADV,N_ED_REF_WK_ENROLL_REAS_ADVANCEMT,Reason for taking courses during reference week: increase advancement opportunities,CG0: D15.6/CGN: D19.6/CGNR: D15.6,For which of the following reasons were you taking courses or enrolled? (Response to: To in...,"Took courses/enrolled in college/university during the week of February 1, 2023",$YES_NO_CODES
1,ACCAR,N_ED_REF_WK_ENROLL_REAS_BEGIN_CAREER,Reason for taking courses during reference week: further education before career,CG0: D15.1/CGN: D19.1/CGNR: D15.1,For which of the following reasons were you taking classes or enrolled? (Response to: To ga...,"Took courses/enrolled in college/university during the week of February 1, 2023",$YES_NO_CODES
2,ACCCEP,N_ED_REF_WK_EMP_ASSISTANCE,School related costs for course taking paid by employer,CG0: D16/CGN: D20/CGNR: D16,Were any of your school-related costs for taking courses paid for by an employer?,Took courses/enrolled in college/university during reference week,$YES_NO_CODES
3,ACCHG,N_ED_REF_WK_ENROLL_REAS_CHG_FLD,Reason for taking courses during reference week: change academic/occupational field,CG0: D15.3/CGN: D19.3/CGNR: D15.3,For which of the following reasons were you taking classes or enrolled? (Response to: To ch...,"Took courses/enrolled in college/university during the week of February 1, 2023",$YES_NO_CODES
4,ACDRG,N_ED_REF_WK_WORK_DEGREE,Type of degree working on during reference week,CG0: D12/CGN: D16/CGNR: D12,Toward what degree were you working?,"Took courses/enrolled in college/university during the week of February 1, 2023",$DEGREE_CODES


,AGE,DGRDG,FACADV,FACBEN,FACCHAL,FACIND,FACLOC,FACRESP,FACSAL,FACSEC,FACSOC,JOBSATIS,LFSTAT,N2OCPRMG,N3OCPR,N3OCPRNG,N3OCPRX,OBSNUM,OCEDRLP,REFYR,SEX_2023,SUPWK,WAPRI,WTSURVY
0,37,1,2,1,2,2,1,2,1,1,2,1,1,1,110610,11,110610,2023054942,3,2023,M,Y,06,5466.1729
1,37,2,2,1,2,1,1,2,1,1,2,2,1,1,121730,12,121730,2023003399,2,2023,F,N,08,46.4920
2,54,1,1,1,2,2,2,1,1,1,1,2,1,7,732550,73,732550,2023007536,1,2023,M,N,13,4520.0847
3,37,1,2,1,2,1,1,2,1,1,2,2,1,7,721510,72,721510,2023036670,2,2023,F,Y,10,9443.6334
4,37,1,3,1,2,2,1,3,1,1,2,L,2,8,999989,98,999989,2023031253,L,2023,F,L,L,7200.3925


,JOBSATIS code,Label
0,1,Very satisfied
1,2,Somewhat satisfied
2,3,Somewhat dissatisfied
3,4,Very dissatisfied
4,L,Logical Skip
5,,Missing


,FAC code,Label
0,1,Very important
1,2,Somewhat important
2,3,Somewhat unimportant
3,4,Not important at all
4,L,Logical Skip
5,,Missing


## 2. Reviewed occupation rules

The specification covers all 129 eligible NSCG categories. It fixes member lists, exclusions and expected counts for this release. Runtime checks read titles and scores from the source files. Table A-2 supplies full occupation labels; SAS labels sometimes omit examples or use shorter wording. Joins use occupation codes.

Only sales engineers and bookkeeping/accounting/auditing clerks require `SUPWK=N`. Exclude software developers, other mathematical scientists, data scientists and other life scientists for missing Work Values; exclude other precollegiate teachers and unclassified occupations for uncertain mapping.

For group profiles, `ONET_SOC_CODE` and `SOC2018_CODE` stay blank. `ONET_PROFILE_ID` identifies the reviewed NSCG profile; full member lists are saved separately. A group with one rated member remains a proxy, not a verified single-occupation match.


In [3]:
# Reviewed membership lists; scores are recalculated from O*NET30.2.
review_spec = json.loads(r'''[
{"code": "110510", "title": "Computer and information research scientists", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["15-1221.00"], "outside_scope": [], "eligible_n": 407, "keep_n": 407, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "110520", "title": "Computer network architects", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the rated .00 profile; omit detailed subtypes.", "candidates": ["15-1241.00"], "outside_scope": [], "eligible_n": 133, "keep_n": 133, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "110540", "title": "Computer support specialists", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Mean of network and user support specialists.", "candidates": ["15-1231.00", "15-1232.00"], "outside_scope": [], "eligible_n": 599, "keep_n": 599, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "110550", "title": "Computer system analysts", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "System/systems wording differs; occupation scope aligns.", "candidates": ["15-1211.00"], "outside_scope": [], "eligible_n": 938, "keep_n": 938, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "110560", "title": "Database administrators", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use Database Administrators; exclude Database Architects.", "candidates": ["15-1242.00"], "outside_scope": [], "eligible_n": 312, "keep_n": 312, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "110570", "title": "Information security analysts", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["15-1212.00"], "outside_scope": [], "eligible_n": 361, "keep_n": 361, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "110580", "title": "Network and computer systems administrators", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["15-1244.00"], "outside_scope": [], "eligible_n": 415, "keep_n": 415, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "110590", "title": "Software developers \u2013 applications and systems software", "decision": "exclude_no_work_values", "method": "no_base_work_values", "proxy_type": "none", "scope_note": "No Work Values for Software Developers .00; do not substitute subtype ratings.", "candidates": ["15-1252.00"], "outside_scope": [], "eligible_n": 4069, "keep_n": 0, "supervisor_excluded_n": 0, "rated_n": 0},
{"code": "110600", "title": "Web developers", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use Web Developers; exclude Web and Digital Interface Designers.", "candidates": ["15-1254.00"], "outside_scope": [], "eligible_n": 325, "keep_n": 325, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "110610", "title": "Other computer and information science occupations", "decision": "include", "method": "group_mean_partial", "proxy_type": "residual_group", "scope_note": "SOC15-1200 minus separately listed computer occupations and programmers; includes database architects, software testers, interface designers and other computer occupations.", "candidates": ["15-1243.00", "15-1253.00", "15-1255.00", "15-1299.00"], "outside_scope": [], "eligible_n": 1162, "keep_n": 1162, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "121720", "title": "Mathematicians", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["15-2021.00"], "outside_scope": [], "eligible_n": 46, "keep_n": 46, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "121730", "title": "Operations research analysts", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["15-2031.00"], "outside_scope": [], "eligible_n": 371, "keep_n": 371, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "121740", "title": "Statisticians", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the rated .00 profile; omit detailed subtypes.", "candidates": ["15-2041.00"], "outside_scope": [], "eligible_n": 322, "keep_n": 322, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "121760", "title": "Other mathematical scientists", "decision": "exclude_no_work_values", "method": "no_base_work_values", "proxy_type": "none", "scope_note": "No Work Values for Mathematical Science Occupations, All Other .00.", "candidates": ["15-2099.00"], "outside_scope": [], "eligible_n": 79, "keep_n": 0, "supervisor_excluded_n": 0, "rated_n": 0},
{"code": "121770", "title": "Data scientists", "decision": "exclude_no_work_values", "method": "no_base_work_values", "proxy_type": "none", "scope_note": "No Work Values for Data Scientists .00; omit rated subtypes.", "candidates": ["15-2051.00"], "outside_scope": [], "eligible_n": 411, "keep_n": 0, "supervisor_excluded_n": 0, "rated_n": 0},
{"code": "182760", "title": "Postsecondary teachers - Computer sciences", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1021.00"], "outside_scope": [], "eligible_n": 154, "keep_n": 154, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "182860", "title": "Postsecondary teachers - Mathematical sciences", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1022.00"], "outside_scope": [], "eligible_n": 322, "keep_n": 322, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "210210", "title": "Agricultural and food scientists", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Mean of animal, food, and soil/plant scientists.", "candidates": ["19-1011.00", "19-1012.00", "19-1013.00"], "outside_scope": [], "eligible_n": 331, "keep_n": 331, "supervisor_excluded_n": 0, "rated_n": 3},
{"code": "220220", "title": "Biochemists and biophysicists", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["19-1021.00"], "outside_scope": [], "eligible_n": 591, "keep_n": 591, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "220230", "title": "Biological scientists (e.g., botanists, ecologists, zoologists)", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "Microbiologists, zoologists and other biological scientists; average rated members.", "candidates": ["19-1022.00", "19-1023.00", "19-1029.00"], "outside_scope": [], "eligible_n": 913, "keep_n": 913, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "220250", "title": "Medical scientists (excluding practitioners)", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Nonclinical medical research: epidemiologists and other medical scientists.", "candidates": ["19-1041.00", "19-1042.00"], "outside_scope": [], "eligible_n": 680, "keep_n": 680, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "220270", "title": "Other life scientists", "decision": "exclude_no_work_values", "method": "no_base_work_values", "proxy_type": "none", "scope_note": "No Work Values for Life Scientists, All Other .00.", "candidates": ["19-1099.00"], "outside_scope": [], "eligible_n": 509, "keep_n": 0, "supervisor_excluded_n": 0, "rated_n": 0},
{"code": "230240", "title": "Conservation scientists and foresters", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Mean of conservation scientists and foresters.", "candidates": ["19-1031.00", "19-1032.00"], "outside_scope": [], "eligible_n": 226, "keep_n": 226, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "282710", "title": "Postsecondary teachers - Agriculture", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1041.00"], "outside_scope": [], "eligible_n": 52, "keep_n": 52, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "282730", "title": "Postsecondary teachers - Biological sciences", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1042.00"], "outside_scope": [], "eligible_n": 360, "keep_n": 360, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "282970", "title": "Postsecondary teachers - Other life and physical sciences (e.g., conservation, etc.)", "decision": "include", "method": "residual_single_proxy", "proxy_type": "residual_single_profile", "scope_note": "Forestry/conservation teaching represents the remaining life/physical-science teaching category; individual teaching fields are unknown.", "candidates": ["25-1043.00"], "outside_scope": ["25-1051.00", "25-1053.00"], "eligible_n": 45, "keep_n": 45, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "311930", "title": "Chemists, except biochemists", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "SOC Chemists also excludes biochemists and biophysicists.", "candidates": ["19-2031.00"], "outside_scope": [], "eligible_n": 962, "keep_n": 962, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "321920", "title": "Atmospheric and space scientists", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["19-2021.00"], "outside_scope": [], "eligible_n": 119, "keep_n": 119, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "321940", "title": "Environmental scientists and specialists", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Environmental science, pollution and environmental health scopes align.", "candidates": ["19-2041.00"], "outside_scope": [], "eligible_n": 319, "keep_n": 319, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "321950", "title": "Geoscientists, including hydrologists and oceanographers", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Mean of geoscientists, including oceanographers, and hydrologists.", "candidates": ["19-2042.00", "19-2043.00"], "outside_scope": [], "eligible_n": 219, "keep_n": 219, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "331960", "title": "Physicists, except biophysicists", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "SOC Physicists also excludes biophysicists.", "candidates": ["19-2012.00"], "outside_scope": [], "eligible_n": 240, "keep_n": 240, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "341980", "title": "Other physical scientists, including astronomersa", "decision": "include", "method": "group_mean_partial", "proxy_type": "residual_group", "scope_note": "Remaining physical sciences: astronomy, materials science and other physical scientists.", "candidates": ["19-2011.00", "19-2032.00", "19-2099.00"], "outside_scope": [], "eligible_n": 236, "keep_n": 236, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "382750", "title": "Postsecondary teachers - Chemistry", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1052.00"], "outside_scope": [], "eligible_n": 162, "keep_n": 162, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "382770", "title": "Postsecondary teachers - Atmospheric, earth, environmental, marine, and space sciences", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Mean of atmospheric/earth/marine/space-science and environmental-science teachers.", "candidates": ["25-1051.00", "25-1053.00"], "outside_scope": [], "eligible_n": 74, "keep_n": 74, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "382890", "title": "Postsecondary teachers - Physics", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1054.00"], "outside_scope": [], "eligible_n": 107, "keep_n": 107, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "412320", "title": "Economists", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the rated .00 profile; omit detailed subtypes.", "candidates": ["19-3011.00"], "outside_scope": [], "eligible_n": 352, "keep_n": 352, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "422350", "title": "Political scientists", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["19-3094.00"], "outside_scope": [], "eligible_n": 90, "keep_n": 90, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "432360", "title": "Psychologists - research and applied (e.g., industrial-organizational, experimental)b", "decision": "include", "method": "single_available_proxy", "proxy_type": "single_available", "scope_note": "Research/applied psychology is separate from health psychology. Only industrial-organizational psychologists have ratings; other psychologists are unrated.", "candidates": ["19-3032.00", "19-3039.00"], "outside_scope": [], "eligible_n": 468, "keep_n": 468, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "442310", "title": "Anthropologists and archeologists", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["19-3091.00"], "outside_scope": [], "eligible_n": 122, "keep_n": 122, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "442370", "title": "Sociologists", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["19-3041.00"], "outside_scope": [], "eligible_n": 66, "keep_n": 66, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "452380", "title": "Other social scientists", "decision": "include", "method": "group_mean_partial", "proxy_type": "residual_group", "scope_note": "Survey research, urban planning, geography and other social sciences; exclude general research assistants.", "candidates": ["19-3022.00", "19-3051.00", "19-3092.00", "19-3099.00"], "outside_scope": ["19-4061.00"], "eligible_n": 603, "keep_n": 603, "supervisor_excluded_n": 0, "rated_n": 3},
{"code": "482780", "title": "Postsecondary teachers - Economics", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1063.00"], "outside_scope": [], "eligible_n": 107, "keep_n": 107, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "482900", "title": "Postsecondary teachers - Political science", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1065.00"], "outside_scope": [], "eligible_n": 68, "keep_n": 68, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "482910", "title": "Postsecondary teachers - Psychology", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1066.00"], "outside_scope": [], "eligible_n": 150, "keep_n": 150, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "482930", "title": "Postsecondary teachers - Sociology", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1067.00"], "outside_scope": [], "eligible_n": 62, "keep_n": 62, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "482980", "title": "Postsecondary teachers - Other social sciences", "decision": "include", "method": "group_mean_partial", "proxy_type": "residual_group", "scope_note": "Anthropology/archeology, area/cultural studies, geography and other social-science teachers.", "candidates": ["25-1061.00", "25-1062.00", "25-1064.00", "25-1069.00"], "outside_scope": [], "eligible_n": 245, "keep_n": 245, "supervisor_excluded_n": 0, "rated_n": 3},
{"code": "510820", "title": "Aeronautical, aerospace, and astronautical engineers", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Aerospace Engineers includes aeronautical and astronautical engineering.", "candidates": ["17-2011.00"], "outside_scope": [], "eligible_n": 679, "keep_n": 679, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "520850", "title": "Chemical engineers", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["17-2041.00"], "outside_scope": [], "eligible_n": 579, "keep_n": 579, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "530860", "title": "Civil engineers, including architectural and sanitary", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Civil Engineers includes architectural engineering and water/sewer facilities.", "candidates": ["17-2051.00"], "outside_scope": [], "eligible_n": 1429, "keep_n": 1429, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "540870", "title": "Computer engineers, hardware", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Both titles specify computer hardware; word order differs.", "candidates": ["17-2061.00"], "outside_scope": [], "eligible_n": 436, "keep_n": 436, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "540890", "title": "Electrical and electronics engineers", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Mean of electrical and electronics engineers; exclude computer hardware engineers.", "candidates": ["17-2071.00", "17-2072.00"], "outside_scope": [], "eligible_n": 1916, "keep_n": 1916, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "550910", "title": "Industrial engineers, including health and safety", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Mean of industrial and health/safety engineers.", "candidates": ["17-2111.00", "17-2112.00"], "outside_scope": [], "eligible_n": 336, "keep_n": 336, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "560940", "title": "Mechanical engineers", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the rated .00 profile; omit detailed subtypes.", "candidates": ["17-2141.00"], "outside_scope": [], "eligible_n": 2106, "keep_n": 2106, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "570840", "title": "Bioengineers and biomedical engineers", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Current .00 titles align; the earlier crosswalk uses Biomedical Engineers.", "candidates": ["17-2031.00"], "outside_scope": [], "eligible_n": 278, "keep_n": 278, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "570900", "title": "Environmental engineers", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["17-2081.00"], "outside_scope": [], "eligible_n": 249, "keep_n": 249, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "570920", "title": "Marine engineers and naval architects", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["17-2121.00"], "outside_scope": [], "eligible_n": 38, "keep_n": 38, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "570930", "title": "Materials engineers", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["17-2131.00"], "outside_scope": [], "eligible_n": 246, "keep_n": 246, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "570950", "title": "Mining and geological engineers, including mining safety engineers", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["17-2151.00"], "outside_scope": [], "eligible_n": 33, "keep_n": 33, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "570960", "title": "Nuclear engineers", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["17-2161.00"], "outside_scope": [], "eligible_n": 94, "keep_n": 94, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "570970", "title": "Petroleum engineers", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["17-2171.00"], "outside_scope": [], "eligible_n": 72, "keep_n": 72, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "570980", "title": "Sales engineers", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Require SUPWK=N; exclude 91 respondents who do not meet this condition.", "candidates": ["41-9031.00"], "outside_scope": [], "eligible_n": 314, "keep_n": 223, "supervisor_excluded_n": 91, "rated_n": 1},
{"code": "570990", "title": "Other engineers, including agriculturalc", "decision": "include", "method": "single_available_proxy", "proxy_type": "single_available", "scope_note": "Agricultural and other engineers; only agricultural engineers have ratings.", "candidates": ["17-2021.00", "17-2199.00"], "outside_scope": [], "eligible_n": 806, "keep_n": 806, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "582800", "title": "Postsecondary teachers - Engineering", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1032.00"], "outside_scope": [], "eligible_n": 294, "keep_n": 294, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "611110", "title": "Diagnosing/treating practitioners (e.g., dentists, optometrists, physicians, psychiatrists, podiatrists, surgeons, veterinarians) ", "decision": "include", "method": "group_mean_partial", "proxy_type": "declared_health_boundary", "scope_note": "Diagnosing/treating group excludes separately listed nursing, pharmacy, therapy and dental hygiene; audiologists are assigned here.", "candidates": ["29-1011.00", "29-1021.00", "29-1022.00", "29-1023.00", "29-1024.00", "29-1029.00", "29-1041.00", "29-1081.00", "29-1131.00", "29-1181.00", "29-1211.00", "29-1212.00", "29-1213.00", "29-1214.00", "29-1215.00", "29-1216.00", "29-1217.00", "29-1218.00", "29-1221.00", "29-1222.00", "29-1223.00", "29-1224.00", "29-1229.00", "29-1241.00", "29-1242.00", "29-1243.00", "29-1249.00", "29-1291.00", "29-1299.00"], "outside_scope": [], "eligible_n": 1564, "keep_n": 1564, "supervisor_excluded_n": 0, "rated_n": 21},
{"code": "611120", "title": "Registered nurses, pharmacists, dieticians, therapists, physician assistants, nurse practitioners", "decision": "include", "method": "group_mean_partial", "proxy_type": "declared_health_boundary", "scope_note": "Nursing, pharmacy, dietetics, therapy, physician assistants and nurse practitioners, including nurse anesthetists and midwives. Audiologists use 611110.", "candidates": ["29-1031.00", "29-1051.00", "29-1071.00", "29-1122.00", "29-1123.00", "29-1124.00", "29-1125.00", "29-1126.00", "29-1127.00", "29-1128.00", "29-1129.00", "29-1141.00", "29-1151.00", "29-1161.00", "29-1171.00"], "outside_scope": ["29-1181.00"], "eligible_n": 4048, "keep_n": 4048, "supervisor_excluded_n": 0, "rated_n": 14},
{"code": "611130", "title": "Health technologists and technicians (e.g., dental hygienists, health record technologists/technicians, licensed practical nurses, medical or laboratory technicians, radiological technicians)", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "Health technical roles named by NSCG, including dental hygiene, laboratory, medical records and health information jobs.", "candidates": ["29-1292.00", "29-2011.00", "29-2012.00", "29-2031.00", "29-2032.00", "29-2033.00", "29-2034.00", "29-2035.00", "29-2036.00", "29-2042.00", "29-2043.00", "29-2051.00", "29-2052.00", "29-2053.00", "29-2055.00", "29-2056.00", "29-2057.00", "29-2061.00", "29-2072.00", "29-2081.00", "29-2091.00", "29-2092.00", "29-2099.00", "29-9021.00"], "outside_scope": [], "eligible_n": 999, "keep_n": 999, "supervisor_excluded_n": 0, "rated_n": 18},
{"code": "611140", "title": "Other health occupations", "decision": "include", "method": "group_mean_partial", "proxy_type": "residual_health_group", "scope_note": "SOC29/31 plus occupational health/safety, minus separately assigned health groups. Health/safety moved from SOC2010 29-901 to SOC2018 19-501.", "candidates": ["19-5011.00", "19-5012.00", "29-9091.00", "29-9092.00", "29-9093.00", "29-9099.00", "31-1121.00", "31-1122.00", "31-1131.00", "31-1132.00", "31-1133.00", "31-2011.00", "31-2012.00", "31-2021.00", "31-2022.00", "31-9011.00", "31-9091.00", "31-9092.00", "31-9093.00", "31-9094.00", "31-9095.00", "31-9096.00", "31-9097.00", "31-9099.00"], "outside_scope": [], "eligible_n": 1754, "keep_n": 1754, "supervisor_excluded_n": 0, "rated_n": 22},
{"code": "611150", "title": "Psychologists - Health services (e.g., clinical, counseling, school psychologists)b", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Mean of clinical/counseling and school psychologists.", "candidates": ["19-3033.00", "19-3034.00"], "outside_scope": [], "eligible_n": 546, "keep_n": 546, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "612870", "title": "Postsecondary teachers - Health", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Mean of health specialties and nursing teachers.", "candidates": ["25-1071.00", "25-1072.00"], "outside_scope": [], "eligible_n": 369, "keep_n": 369, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "621420", "title": "Computer and information systems managers", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the broader management profile; NSCG specifies managers of other managers.", "candidates": ["11-3021.00"], "outside_scope": [], "eligible_n": 727, "keep_n": 727, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "621430", "title": "Engineering managers", "decision": "include", "method": "single_SOC_broader_profile", "proxy_type": "broader_profile", "scope_note": "Use the broader Architectural and Engineering Managers profile.", "candidates": ["11-9041.00"], "outside_scope": [], "eligible_n": 1250, "keep_n": 1250, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "621440", "title": "Medical and health services managers", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the broader management profile; NSCG specifies managers of other managers.", "candidates": ["11-9111.00"], "outside_scope": [], "eligible_n": 743, "keep_n": 743, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "621450", "title": "Natural sciences managers", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the rated .00 profile; omit detailed subtypes.", "candidates": ["11-9121.00"], "outside_scope": [], "eligible_n": 396, "keep_n": 396, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "632530", "title": "Teachers: Secondary - computer, math, or sciences", "decision": "include", "method": "single_SOC_broader_profile", "proxy_type": "broader_profile", "scope_note": "Use general secondary-school teaching; possible vocational teaching is a mapping assumption.", "candidates": ["25-2031.00"], "outside_scope": [], "eligible_n": 1397, "keep_n": 1397, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "632540", "title": "Teachers: Secondary - social sciences", "decision": "include", "method": "single_SOC_broader_profile", "proxy_type": "broader_profile", "scope_note": "General secondary-school teaching includes social sciences; no subject-specific profile is required.", "candidates": ["25-2031.00"], "outside_scope": [], "eligible_n": 363, "keep_n": 363, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "640260", "title": "Technologists and technicians in the biological/life sciences", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Agricultural, food, biological, and forest/conservation technicians.", "candidates": ["19-4012.00", "19-4013.00", "19-4021.00", "19-4071.00"], "outside_scope": [], "eligible_n": 663, "keep_n": 663, "supervisor_excluded_n": 0, "rated_n": 4},
{"code": "640530", "title": "Computer programmers (business, scientific, process control)", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Listed programming fields use Computer Programmers.", "candidates": ["15-1251.00"], "outside_scope": [], "eligible_n": 700, "keep_n": 700, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "641000", "title": "Electrical, electronic, industrial, and mechanical technicians", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Electrical/electronic, industrial, mechanical and electromechanical technicians.", "candidates": ["17-3023.00", "17-3024.00", "17-3026.00", "17-3027.00"], "outside_scope": [], "eligible_n": 1183, "keep_n": 1183, "supervisor_excluded_n": 0, "rated_n": 4},
{"code": "641010", "title": "Drafting occupations, including computer drafting", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "Architectural/civil, electrical/electronic, mechanical and other drafters.", "candidates": ["17-3011.00", "17-3012.00", "17-3013.00", "17-3019.00"], "outside_scope": [], "eligible_n": 182, "keep_n": 182, "supervisor_excluded_n": 0, "rated_n": 3},
{"code": "641020", "title": "Surveying and mapping technicians", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["17-3031.00"], "outside_scope": [], "eligible_n": 66, "keep_n": 66, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "641030", "title": "Other engineering technologists and technicians", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "Aerospace, civil, environmental, calibration and other engineering technicians; exclude separately listed technician groups.", "candidates": ["17-3021.00", "17-3022.00", "17-3025.00", "17-3028.00", "17-3029.00"], "outside_scope": [], "eligible_n": 1050, "keep_n": 1050, "supervisor_excluded_n": 0, "rated_n": 3},
{"code": "641040", "title": "Surveyors, cartographers, photogrammetrists", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Mean of surveyors and cartographers/photogrammetrists.", "candidates": ["17-1021.00", "17-1022.00"], "outside_scope": [], "eligible_n": 85, "keep_n": 85, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "641970", "title": "Technologists and technicians in the physical sciences", "decision": "include", "method": "group_mean_partial", "proxy_type": "residual_group", "scope_note": "Chemical, environmental, geological, hydrologic, nuclear, forensic and other science technicians. The unrated residual spans life, physical and social science.", "candidates": ["19-4031.00", "19-4042.00", "19-4043.00", "19-4044.00", "19-4051.00", "19-4092.00", "19-4099.00"], "outside_scope": [], "eligible_n": 117, "keep_n": 117, "supervisor_excluded_n": 0, "rated_n": 5},
{"code": "650810", "title": "Architects, except naval", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Mean of architects, excluding landscape/naval, and landscape architects.", "candidates": ["17-1011.00", "17-1012.00"], "outside_scope": [], "eligible_n": 436, "keep_n": 436, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "651710", "title": "Actuaries", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Occupation titles and scope align.", "candidates": ["15-2011.00"], "outside_scope": [], "eligible_n": 176, "keep_n": 176, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "711410", "title": "Top-executives (e.g., chief executives, general and operations managers, legislators)", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Chief executives, general/operations managers and legislators.", "candidates": ["11-1011.00", "11-1021.00", "11-1031.00"], "outside_scope": [], "eligible_n": 3673, "keep_n": 3673, "supervisor_excluded_n": 0, "rated_n": 3},
{"code": "711460", "title": "Education and childcare administrators", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "Preschool/childcare, school, college and other education administrators.", "candidates": ["11-9031.00", "11-9032.00", "11-9033.00", "11-9039.00"], "outside_scope": [], "eligible_n": 390, "keep_n": 390, "supervisor_excluded_n": 0, "rated_n": 3},
{"code": "711470", "title": "Other mid-level managers", "decision": "include", "method": "group_mean_partial", "proxy_type": "residual_group", "scope_note": "Management occupations minus top executives and separately listed computer, engineering, health, natural-science and education managers.", "candidates": ["11-2011.00", "11-2021.00", "11-2022.00", "11-2032.00", "11-2033.00", "11-3012.00", "11-3013.00", "11-3031.00", "11-3051.00", "11-3061.00", "11-3071.00", "11-3111.00", "11-3121.00", "11-3131.00", "11-9013.00", "11-9021.00", "11-9051.00", "11-9071.00", "11-9072.00", "11-9081.00", "11-9131.00", "11-9141.00", "11-9151.00", "11-9161.00", "11-9171.00", "11-9179.00", "11-9199.00"], "outside_scope": [], "eligible_n": 2184, "keep_n": 2184, "supervisor_excluded_n": 0, "rated_n": 21},
{"code": "721510", "title": "Financial specialists (e.g., accountants, auditors)", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "All 15 detailed SOC13-2000 occupations; 13-2099 is only the residual member.", "candidates": ["13-2011.00", "13-2022.00", "13-2023.00", "13-2031.00", "13-2041.00", "13-2051.00", "13-2052.00", "13-2053.00", "13-2054.00", "13-2061.00", "13-2071.00", "13-2072.00", "13-2081.00", "13-2082.00", "13-2099.00"], "outside_scope": [], "eligible_n": 2038, "keep_n": 2038, "supervisor_excluded_n": 0, "rated_n": 11},
{"code": "721520", "title": "Personnel, training, and labor relations specialists", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Human resources, labor relations, compensation/benefits and training specialists.", "candidates": ["13-1071.00", "13-1075.00", "13-1141.00", "13-1151.00"], "outside_scope": [], "eligible_n": 769, "keep_n": 769, "supervisor_excluded_n": 0, "rated_n": 4},
{"code": "721530", "title": "Other management related occupations (e.g., buyers and purchasing agents, claims adjusters, compliance officers, logisticians, management analysts, marketing specialists) ", "decision": "include", "method": "group_mean_partial", "proxy_type": "residual_group", "scope_note": "Business operations minus the four separately listed personnel/training occupations. Finance is separate; farm labor contractors remain included.", "candidates": ["13-1011.00", "13-1021.00", "13-1022.00", "13-1023.00", "13-1031.00", "13-1032.00", "13-1041.00", "13-1051.00", "13-1074.00", "13-1081.00", "13-1082.00", "13-1111.00", "13-1121.00", "13-1131.00", "13-1161.00", "13-1199.00"], "outside_scope": [], "eligible_n": 3381, "keep_n": 3381, "supervisor_excluded_n": 0, "rated_n": 14},
{"code": "732510", "title": "Teachers: Preschool and kindergarten ", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Mean of preschool and kindergarten teachers.", "candidates": ["25-2011.00", "25-2012.00"], "outside_scope": [], "eligible_n": 228, "keep_n": 228, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "732520", "title": "Teachers: Elementary and middle school ", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Elementary, general middle-school and middle-school vocational teachers.", "candidates": ["25-2021.00", "25-2022.00", "25-2023.00"], "outside_scope": [], "eligible_n": 1086, "keep_n": 1086, "supervisor_excluded_n": 0, "rated_n": 3},
{"code": "732550", "title": "Teachers: Secondary - other subjects", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "General and vocational secondary-school teachers represent other subjects.", "candidates": ["25-2031.00", "25-2032.00"], "outside_scope": [], "eligible_n": 650, "keep_n": 650, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "732560", "title": "Teachers: Special education - primary and secondary school", "decision": "include", "method": "group_mean_partial", "proxy_type": "school_stage_assumption", "scope_note": "Kindergarten through secondary special education; exclude preschool and unspecified levels. Treat kindergarten as primary education.", "candidates": ["25-2055.00", "25-2056.00", "25-2057.00", "25-2058.00"], "outside_scope": ["25-2051.00", "25-2059.00"], "eligible_n": 418, "keep_n": 418, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "732570", "title": "Teachers: Other precollegiate area", "decision": "exclude_uncertain_mapping", "method": "excluded_scope", "proxy_type": "none", "scope_note": "Teaching subtypes are unspecified. SOC25-2000 is too broad; adult education cannot represent the whole category.", "candidates": ["25-3011.00", "25-3031.00", "25-3099.00"], "outside_scope": ["25-9031.00", "25-9042.00", "25-9043.00"], "eligible_n": 171, "keep_n": 0, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "742720", "title": "Postsecondary teachers - Art, drama, and music", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1121.00"], "outside_scope": [], "eligible_n": 131, "keep_n": 131, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "742740", "title": "Postsecondary teachers - Business", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1011.00"], "outside_scope": [], "eligible_n": 207, "keep_n": 207, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "742790", "title": "Postsecondary teachers - Education", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1081.00"], "outside_scope": [], "eligible_n": 180, "keep_n": 180, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "742810", "title": "Postsecondary teachers - English language and literature", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1123.00"], "outside_scope": [], "eligible_n": 135, "keep_n": 135, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "742820", "title": "Postsecondary teachers - Foreign language and literature", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1124.00"], "outside_scope": [], "eligible_n": 137, "keep_n": 137, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "742830", "title": "Postsecondary teachers - History", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1125.00"], "outside_scope": [], "eligible_n": 80, "keep_n": 80, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "742880", "title": "Postsecondary teachers - Recreation and fitness studies", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Use the subject-specific postsecondary teaching profile.", "candidates": ["25-1193.00"], "outside_scope": [], "eligible_n": 12, "keep_n": 12, "supervisor_excluded_n": 0, "rated_n": 1},
{"code": "742990", "title": "Postsecondary teachers - Other postsecondary fields", "decision": "include", "method": "group_mean_partial", "proxy_type": "residual_group", "scope_note": "SOC25-1000 minus separately listed teaching fields; exclude postsecondary teaching assistants.", "candidates": ["25-1031.00", "25-1082.00", "25-1111.00", "25-1112.00", "25-1113.00", "25-1122.00", "25-1126.00", "25-1192.00", "25-1194.00", "25-1199.00"], "outside_scope": ["25-9044.00"], "eligible_n": 357, "keep_n": 357, "supervisor_excluded_n": 0, "rated_n": 9},
{"code": "750400", "title": "Clergy and other religious workers", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "Clergy, religious activity directors and other religious workers.", "candidates": ["21-2011.00", "21-2021.00", "21-2099.00"], "outside_scope": [], "eligible_n": 298, "keep_n": 298, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "750700", "title": "Counselors (e.g., educational, vocational, mental health, and substance abuse)", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "Educational/career, mental-health, addiction, rehabilitation and other counselors.", "candidates": ["21-1011.00", "21-1012.00", "21-1013.00", "21-1014.00", "21-1015.00", "21-1019.00"], "outside_scope": [], "eligible_n": 1190, "keep_n": 1190, "supervisor_excluded_n": 0, "rated_n": 5},
{"code": "752400", "title": "Social workers", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "Child/family, healthcare, mental-health and other social workers.", "candidates": ["21-1021.00", "21-1022.00", "21-1023.00", "21-1029.00"], "outside_scope": [], "eligible_n": 446, "keep_n": 446, "supervisor_excluded_n": 0, "rated_n": 3},
{"code": "752410", "title": "Miscellaneous community and social service specialists (e.g., probation officers and correctional treatment specialists)", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "SOC21-109 community and social service specialists.", "candidates": ["21-1091.00", "21-1092.00", "21-1093.00", "21-1094.00", "21-1099.00"], "outside_scope": [], "eligible_n": 659, "keep_n": 659, "supervisor_excluded_n": 0, "rated_n": 4},
{"code": "762000", "title": "Sales representatives: services (e.g., insurance, securities, real estate, and business services)", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "Insurance, securities, service and real-estate sales; exclude separately listed sales engineers.", "candidates": ["41-3011.00", "41-3021.00", "41-3031.00", "41-3041.00", "41-3091.00", "41-9021.00", "41-9022.00"], "outside_scope": [], "eligible_n": 1147, "keep_n": 1147, "supervisor_excluded_n": 0, "rated_n": 6},
{"code": "762010", "title": "Sales representatives: wholesale and manufacturing (e.g., industrial machinery/equipment/supplies)", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Wholesale/manufacturing sales of technical/scientific and other products.", "candidates": ["41-4011.00", "41-4012.00"], "outside_scope": [], "eligible_n": 506, "keep_n": 506, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "762020", "title": "Retail sales workers (e.g., furnishings, clothing, motor vehicles, cosmetics)", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Retail sales, cashiers, counter/parts sales and retail supervisors.", "candidates": ["41-1011.00", "41-2011.00", "41-2012.00", "41-2021.00", "41-2022.00", "41-2031.00"], "outside_scope": [], "eligible_n": 594, "keep_n": 594, "supervisor_excluded_n": 0, "rated_n": 6},
{"code": "762030", "title": "Miscellaneous sales and related workers", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "Nonretail sales supervisors, demonstrators, models, telemarketers and other sales workers.", "candidates": ["41-1012.00", "41-9011.00", "41-9012.00", "41-9041.00", "41-9091.00", "41-9099.00"], "outside_scope": [], "eligible_n": 851, "keep_n": 851, "supervisor_excluded_n": 0, "rated_n": 5},
{"code": "770100", "title": "Writers, editors, public relations specialists, artists, entertainers, broadcasters, and historiansd", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "SOC27 arts/design/entertainment/sports/media plus historians; a broad occupational background.", "candidates": ["19-3093.00", "27-1011.00", "27-1012.00", "27-1013.00", "27-1014.00", "27-1019.00", "27-1021.00", "27-1022.00", "27-1023.00", "27-1024.00", "27-1025.00", "27-1026.00", "27-1027.00", "27-1029.00", "27-2011.00", "27-2012.00", "27-2021.00", "27-2022.00", "27-2023.00", "27-2031.00", "27-2032.00", "27-2041.00", "27-2042.00", "27-2091.00", "27-2099.00", "27-3011.00", "27-3023.00", "27-3031.00", "27-3041.00", "27-3042.00", "27-3043.00", "27-3091.00", "27-3092.00", "27-3099.00", "27-4011.00", "27-4012.00", "27-4014.00", "27-4015.00", "27-4021.00", "27-4031.00", "27-4032.00", "27-4099.00"], "outside_scope": [], "eligible_n": 1206, "keep_n": 1206, "supervisor_excluded_n": 0, "rated_n": 35},
{"code": "780310", "title": "Bookkeeping, accounting, and auditing clerks", "decision": "include", "method": "single_SOC", "proxy_type": "none", "scope_note": "Require SUPWK=N; exclude 77 respondents who do not meet this condition.", "candidates": ["43-3031.00"], "outside_scope": [], "eligible_n": 455, "keep_n": 378, "supervisor_excluded_n": 77, "rated_n": 1},
{"code": "780320", "title": "Secretaries and administrative assistants, and receptionists", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "Four secretary/administrative-assistant occupations plus receptionists.", "candidates": ["43-4171.00", "43-6011.00", "43-6012.00", "43-6013.00", "43-6014.00"], "outside_scope": [], "eligible_n": 574, "keep_n": 574, "supervisor_excluded_n": 0, "rated_n": 5},
{"code": "780330", "title": "Other office and administrative support workers", "decision": "include", "method": "group_mean_partial", "proxy_type": "residual_group", "scope_note": "SOC43 minus bookkeeping, secretaries/administrative assistants and receptionists; office supervisors remain included.", "candidates": ["43-1011.00", "43-2011.00", "43-2021.00", "43-2099.00", "43-3011.00", "43-3021.00", "43-3041.00", "43-3051.00", "43-3061.00", "43-3071.00", "43-3099.00", "43-4011.00", "43-4021.00", "43-4031.00", "43-4041.00", "43-4051.00", "43-4061.00", "43-4071.00", "43-4081.00", "43-4111.00", "43-4121.00", "43-4131.00", "43-4141.00", "43-4151.00", "43-4161.00", "43-4181.00", "43-4199.00", "43-5011.00", "43-5021.00", "43-5031.00", "43-5032.00", "43-5041.00", "43-5051.00", "43-5052.00", "43-5053.00", "43-5061.00", "43-5071.00", "43-5111.00", "43-9021.00", "43-9022.00", "43-9031.00", "43-9041.00", "43-9051.00", "43-9061.00", "43-9071.00", "43-9081.00", "43-9111.00", "43-9199.00"], "outside_scope": [], "eligible_n": 1955, "keep_n": 1955, "supervisor_excluded_n": 0, "rated_n": 44},
{"code": "781100", "title": "Farming, fishing, and forestry occupations", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "SOC45 farming, fishing and forestry occupations.", "candidates": ["45-1011.00", "45-2011.00", "45-2021.00", "45-2041.00", "45-2091.00", "45-2092.00", "45-2093.00", "45-2099.00", "45-3031.00", "45-4011.00", "45-4021.00", "45-4022.00", "45-4023.00", "45-4029.00"], "outside_scope": [], "eligible_n": 593, "keep_n": 593, "supervisor_excluded_n": 0, "rated_n": 12},
{"code": "781200", "title": "Lawyers, judges, and related workers", "decision": "include", "method": "group_mean_complete", "proxy_type": "none", "scope_note": "SOC23-1000 lawyers, judges and related workers.", "candidates": ["23-1011.00", "23-1012.00", "23-1021.00", "23-1022.00", "23-1023.00"], "outside_scope": [], "eligible_n": 1192, "keep_n": 1192, "supervisor_excluded_n": 0, "rated_n": 5},
{"code": "781250", "title": "Legal support workers (e.g., paralegals, legal assistants)", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "SOC23-2000 legal support occupations.", "candidates": ["23-2011.00", "23-2093.00", "23-2099.00"], "outside_scope": [], "eligible_n": 287, "keep_n": 287, "supervisor_excluded_n": 0, "rated_n": 2},
{"code": "781300", "title": "Librarians, archivists, and curators", "decision": "include", "method": "group_mean_complete", "proxy_type": "broad_family", "scope_note": "All five SOC25-4000 occupations, including library and museum technicians.", "candidates": ["25-4011.00", "25-4012.00", "25-4013.00", "25-4022.00", "25-4031.00"], "outside_scope": [], "eligible_n": 235, "keep_n": 235, "supervisor_excluded_n": 0, "rated_n": 5},
{"code": "782210", "title": "Food preparation and service occupations (e.g., cooks, waitresses, bartenders)", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "SOC35 food preparation and serving occupations.", "candidates": ["35-1011.00", "35-1012.00", "35-2011.00", "35-2012.00", "35-2013.00", "35-2014.00", "35-2015.00", "35-2019.00", "35-2021.00", "35-3011.00", "35-3023.00", "35-3031.00", "35-3041.00", "35-9011.00", "35-9021.00", "35-9031.00", "35-9099.00"], "outside_scope": [], "eligible_n": 346, "keep_n": 346, "supervisor_excluded_n": 0, "rated_n": 15},
{"code": "782220", "title": "Protective services occupations (e.g., firefighters, police, guards, wardens, park rangers)", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "SOC33 protective service occupations.", "candidates": ["33-1011.00", "33-1012.00", "33-1021.00", "33-1091.00", "33-1099.00", "33-2011.00", "33-2021.00", "33-2022.00", "33-3011.00", "33-3012.00", "33-3021.00", "33-3031.00", "33-3041.00", "33-3051.00", "33-3052.00", "33-9011.00", "33-9021.00", "33-9031.00", "33-9032.00", "33-9091.00", "33-9092.00", "33-9093.00", "33-9094.00", "33-9099.00"], "outside_scope": [], "eligible_n": 419, "keep_n": 419, "supervisor_excluded_n": 0, "rated_n": 20},
{"code": "782230", "title": "Other service occupations, except health (e.g., personal care service occupations, building and grounds cleaning and maintenance occupations, etc.)", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "SOC37 building/grounds maintenance plus SOC39 personal care; exclude healthcare support.", "candidates": ["37-1011.00", "37-1012.00", "37-2011.00", "37-2012.00", "37-2019.00", "37-2021.00", "37-3011.00", "37-3012.00", "37-3013.00", "37-3019.00", "39-1013.00", "39-1014.00", "39-1022.00", "39-2011.00", "39-2021.00", "39-3011.00", "39-3012.00", "39-3019.00", "39-3021.00", "39-3031.00", "39-3091.00", "39-3092.00", "39-3093.00", "39-3099.00", "39-4011.00", "39-4012.00", "39-4021.00", "39-4031.00", "39-5011.00", "39-5012.00", "39-5091.00", "39-5092.00", "39-5093.00", "39-5094.00", "39-6011.00", "39-6012.00", "39-7011.00", "39-7012.00", "39-9011.00", "39-9031.00", "39-9032.00", "39-9041.00", "39-9099.00"], "outside_scope": [], "eligible_n": 739, "keep_n": 739, "supervisor_excluded_n": 0, "rated_n": 36},
{"code": "783000", "title": "Other teachers and instructors (e.g., private tutors, dance or flying instructors, martial arts instructors)", "decision": "include", "method": "group_mean_partial", "proxy_type": "cross_family_teaching", "scope_note": "Tutoring, self-enrichment, farm/home education, sports/fitness and flight instruction. Pilot and coaching profiles also include non-teaching duties.", "candidates": ["25-3021.00", "25-3041.00", "25-3099.00", "25-9021.00", "27-2022.00", "39-9031.00", "53-2011.00", "53-2012.00"], "outside_scope": ["25-9049.00", "25-9099.00"], "eligible_n": 661, "keep_n": 661, "supervisor_excluded_n": 0, "rated_n": 7},
{"code": "784010", "title": "Construction and extraction occupations", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "SOC47 construction and extraction occupations.", "candidates": ["47-1011.00", "47-2011.00", "47-2021.00", "47-2022.00", "47-2031.00", "47-2041.00", "47-2042.00", "47-2043.00", "47-2044.00", "47-2051.00", "47-2053.00", "47-2061.00", "47-2071.00", "47-2072.00", "47-2073.00", "47-2081.00", "47-2082.00", "47-2111.00", "47-2121.00", "47-2131.00", "47-2132.00", "47-2141.00", "47-2142.00", "47-2151.00", "47-2152.00", "47-2161.00", "47-2171.00", "47-2181.00", "47-2211.00", "47-2221.00", "47-2231.00", "47-3011.00", "47-3012.00", "47-3013.00", "47-3014.00", "47-3015.00", "47-3016.00", "47-3019.00", "47-4011.00", "47-4021.00", "47-4031.00", "47-4041.00", "47-4051.00", "47-4061.00", "47-4071.00", "47-4091.00", "47-4099.00", "47-5011.00", "47-5012.00", "47-5013.00", "47-5022.00", "47-5023.00", "47-5032.00", "47-5041.00", "47-5043.00", "47-5044.00", "47-5049.00", "47-5051.00", "47-5071.00", "47-5081.00", "47-5099.00"], "outside_scope": [], "eligible_n": 229, "keep_n": 229, "supervisor_excluded_n": 0, "rated_n": 57},
{"code": "784020", "title": "Installation, maintenance, and repair occupations", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "SOC49 installation, maintenance and repair occupations.", "candidates": ["49-1011.00", "49-2011.00", "49-2021.00", "49-2022.00", "49-2091.00", "49-2092.00", "49-2093.00", "49-2094.00", "49-2095.00", "49-2096.00", "49-2097.00", "49-2098.00", "49-3011.00", "49-3021.00", "49-3022.00", "49-3023.00", "49-3031.00", "49-3041.00", "49-3042.00", "49-3043.00", "49-3051.00", "49-3052.00", "49-3053.00", "49-3091.00", "49-3092.00", "49-3093.00", "49-9011.00", "49-9012.00", "49-9021.00", "49-9031.00", "49-9041.00", "49-9043.00", "49-9044.00", "49-9045.00", "49-9051.00", "49-9052.00", "49-9061.00", "49-9062.00", "49-9063.00", "49-9064.00", "49-9069.00", "49-9071.00", "49-9081.00", "49-9091.00", "49-9092.00", "49-9094.00", "49-9095.00", "49-9096.00", "49-9097.00", "49-9098.00", "49-9099.00"], "outside_scope": [], "eligible_n": 206, "keep_n": 206, "supervisor_excluded_n": 0, "rated_n": 49},
{"code": "784030", "title": "Production occupations (e.g., assemblers, bakers, metal workers, tailors, woodworkers)", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "SOC51 production occupations.", "candidates": ["51-1011.00", "51-2011.00", "51-2021.00", "51-2022.00", "51-2023.00", "51-2031.00", "51-2041.00", "51-2051.00", "51-2061.00", "51-2092.00", "51-2099.00", "51-3011.00", "51-3021.00", "51-3022.00", "51-3023.00", "51-3091.00", "51-3092.00", "51-3093.00", "51-3099.00", "51-4021.00", "51-4022.00", "51-4023.00", "51-4031.00", "51-4032.00", "51-4033.00", "51-4034.00", "51-4035.00", "51-4041.00", "51-4051.00", "51-4052.00", "51-4061.00", "51-4062.00", "51-4071.00", "51-4072.00", "51-4081.00", "51-4111.00", "51-4121.00", "51-4122.00", "51-4191.00", "51-4192.00", "51-4193.00", "51-4194.00", "51-4199.00", "51-5111.00", "51-5112.00", "51-5113.00", "51-6011.00", "51-6021.00", "51-6031.00", "51-6041.00", "51-6042.00", "51-6051.00", "51-6052.00", "51-6061.00", "51-6062.00", "51-6063.00", "51-6064.00", "51-6091.00", "51-6092.00", "51-6093.00", "51-6099.00", "51-7011.00", "51-7021.00", "51-7031.00", "51-7032.00", "51-7041.00", "51-7042.00", "51-7099.00", "51-8011.00", "51-8012.00", "51-8013.00", "51-8021.00", "51-8031.00", "51-8091.00", "51-8092.00", "51-8093.00", "51-8099.00", "51-9011.00", "51-9012.00", "51-9021.00", "51-9022.00", "51-9023.00", "51-9031.00", "51-9032.00", "51-9041.00", "51-9051.00", "51-9061.00", "51-9071.00", "51-9081.00", "51-9082.00", "51-9083.00", "51-9111.00", "51-9123.00", "51-9124.00", "51-9141.00", "51-9151.00", "51-9161.00", "51-9162.00", "51-9191.00", "51-9192.00", "51-9193.00", "51-9194.00", "51-9195.00", "51-9196.00", "51-9197.00", "51-9198.00", "51-9199.00"], "outside_scope": [], "eligible_n": 383, "keep_n": 383, "supervisor_excluded_n": 0, "rated_n": 100},
{"code": "784050", "title": "Transportation and material moving occupations", "decision": "include", "method": "group_mean_partial", "proxy_type": "none", "scope_note": "SOC53 transportation and material-moving occupations.", "candidates": ["53-1041.00", "53-1042.00", "53-1043.00", "53-1044.00", "53-1049.00", "53-2011.00", "53-2012.00", "53-2021.00", "53-2022.00", "53-2031.00", "53-3011.00", "53-3031.00", "53-3032.00", "53-3033.00", "53-3051.00", "53-3052.00", "53-3053.00", "53-3054.00", "53-3099.00", "53-4011.00", "53-4013.00", "53-4022.00", "53-4031.00", "53-4041.00", "53-4099.00", "53-5011.00", "53-5021.00", "53-5022.00", "53-5031.00", "53-6011.00", "53-6021.00", "53-6031.00", "53-6032.00", "53-6041.00", "53-6051.00", "53-6061.00", "53-6099.00", "53-7011.00", "53-7021.00", "53-7031.00", "53-7041.00", "53-7051.00", "53-7061.00", "53-7062.00", "53-7063.00", "53-7064.00", "53-7065.00", "53-7071.00", "53-7072.00", "53-7073.00", "53-7081.00", "53-7121.00", "53-7199.00"], "outside_scope": [], "eligible_n": 402, "keep_n": 402, "supervisor_excluded_n": 0, "rated_n": 43},
{"code": "785000", "title": "Other occupations (not elsewhere classified)", "decision": "exclude_uncertain_mapping", "method": "excluded_scope", "proxy_type": "none", "scope_note": "No identifiable occupation group; do not substitute an arbitrary All Other profile or an all-occupation mean.", "candidates": [], "outside_scope": [], "eligible_n": 377, "keep_n": 0, "supervisor_excluded_n": 0, "rated_n": 0}
]''')
rules = pd.DataFrame(review_spec)
assert len(rules) == 129 and rules.code.is_unique
assert rules.decision.value_counts().to_dict() == {
    'include': 123, 'exclude_no_work_values': 4, 'exclude_uncertain_mapping': 2}
assert rules.keep_n.sum() == 74016 and rules.eligible_n.sum() == 79800
assert rules.supervisor_excluded_n.sum() == 168
assert set(rules.loc[rules.supervisor_excluded_n.gt(0), 'code']) == {'570980', '780310'}

hierarchy = pd.read_excel(SOURCE['hierarchy'], header=None).iloc[5:, [6, 7]].dropna()
hierarchy.columns = ['code', 'title']
hierarchy.code = hierarchy.code.astype(str).str.replace(r'\.0$', '', regex=True)
assert len(hierarchy) == 129 and hierarchy.code.is_unique
published_titles = hierarchy.set_index('code').title
normalize = lambda x: re.sub(r'[^a-z0-9]+', ' ', str(x).casefold()).strip()
assert set(rules.code) == set(published_titles.index)
for r in rules.itertuples():
    assert normalize(r.title) == normalize(published_titles[r.code])
assert set(rules.code) <= set(labels_for('N3OCPR'))
rules['sas_title'] = rules.code.map(labels_for('N3OCPR'))
assert rules.sas_title.notna().all() and rules.sas_title.str.strip().ne('').all()

occupations = pd.read_csv(SOURCE['occupations'], sep='	').set_index('O*NET-SOC Code')
official = pd.read_excel(SOURCE['crosswalk'], header=3, dtype=str).set_index('O*NET-SOC 2019 Code')
assert occupations.index.is_unique and official.index.is_unique
all_codes = {c for r in review_spec for c in r['candidates'] + r['outside_scope']}
assert all(c.endswith('.00') for c in all_codes)
assert all_codes <= set(occupations.index) and all_codes <= set(official.index)
assert all(official.at[c, '2018 SOC Code'] == c[:7] for c in all_codes)
for r in review_spec:
    assert len(r['candidates']) == len(set(r['candidates']))
    assert not set(r['candidates']) & set(r['outside_scope'])
display(rules.groupby('decision').agg(categories=('code', 'size'), people=('eligible_n', 'sum')))


,categories,people
decision,,
exclude_no_work_values,4,5068
exclude_uncertain_mapping,2,548
include,123,74184


## 3. Build one profile per included NSCG category

Use the six `EX` Work Values on their original 1–7 scale. A member must have all six ratings to enter any mean. Calculate all values over the same member set; use equal occupation weights, not respondent or employment weights.

`MAPPING_COVERAGE` records full/partial rating coverage. `MAPPING_IS_PROXY` and `MAPPING_PROXY_TYPE` record additional assumptions. Rating dates describe the underlying O*NET records, not the survey year.


In [4]:
values_raw = pd.read_csv(SOURCE['values'], sep='	')
content = pd.read_csv(SOURCE['content'], sep='	')
scales = pd.read_csv(SOURCE['scales'], sep='	')
value_definitions = content.loc[content['Element ID'].str.fullmatch(r'1\.B\.2\.[a-f]')].sort_values('Element ID').copy()
assert len(value_definitions) == 6 and value_definitions['Element ID'].is_unique
value_definitions['Output column'] = 'ONET_WV_' + value_definitions['Element Name'].str.lower().str.replace(r'[^a-z0-9]+', '_', regex=True)
value_columns = value_definitions['Output column'].tolist()
element_columns = value_definitions.set_index('Element ID')['Output column'].to_dict()
ex_scale = scales.loc[scales['Scale ID'].eq('EX')].squeeze()
assert (ex_scale.Minimum, ex_scale.Maximum) == (1, 7)
value_rows = values_raw.loc[values_raw['Scale ID'].eq('EX') &
    values_raw['O*NET-SOC Code'].isin(all_codes) & values_raw['Element ID'].isin(element_columns)].copy()
assert not value_rows.duplicated(['O*NET-SOC Code', 'Element ID']).any()
assert value_rows['Data Value'].between(1, 7).all()
value_rows['rating_month'] = pd.to_datetime(value_rows.Date, format='%m/%Y', errors='raise')
wide = value_rows.pivot(index='O*NET-SOC Code', columns='Element ID', values='Data Value')
wide = wide.reindex(index=sorted(all_codes), columns=list(element_columns)).rename(columns=element_columns)
complete = wide.notna().all(axis=1) & wide.ge(1).all(axis=1) & wide.le(7).all(axis=1)
profile_records, candidate_records = [], []
for r in review_spec:
    members = r['candidates']
    rated = [c for c in members if complete.at[c]]
    assert len(rated) == r['rated_n'], r['code']
    included = r['decision'] == 'include'
    for c in members + r['outside_scope']:
        in_scope = c in members
        used = included and in_scope and bool(complete.at[c])
        reason = ('Include' if used else 'Exclude (outside scope)' if not in_scope else
                  'Exclude (uncertain mapping)' if r['decision']=='exclude_uncertain_mapping' else
                  'Exclude (no Work Values)')
        candidate_records.append({'N3OCPR':r['code'], 'NSCG_OCC_TITLE':r['title'],
            'ONET_CODE':c, 'ONET_TITLE':occupations.at[c, 'Title'],
            'IN_DEFINED_GROUP':in_scope, 'COMPLETE_RATINGS':bool(complete.at[c]),
            'USED_IN_PROFILE':used, 'DECISION':reason, **wide.loc[c].to_dict()})
    if not included:
        continue
    assert rated, r['code']
    selected = wide.loc[rated, value_columns]
    raw_ratings = value_rows.loc[value_rows['O*NET-SOC Code'].isin(rated)]
    single = len(members) == 1
    profile_records.append({
        'ONET_PROFILE_ID':'NSCG_' + r['code'],
        'SOC2018_CODE':members[0][:7] if single else '',
        'ONET_SOC_CODE':members[0] if single else '',
        'ONET_OCC_TITLE':occupations.at[members[0], 'Title'] if single else 'Group mean: ' + r['title'],
        'MAPPING_TYPE':r['method'], 'MAPPING_COVERAGE':'full' if len(rated)==len(members) else 'partial',
        'MAPPING_IS_PROXY':r['proxy_type']!='none', 'MAPPING_PROXY_TYPE':r['proxy_type'],
        'ONET_CANDIDATE_COUNT':len(members), 'ONET_RATED_COUNT':len(rated),
        'ONET_RATING_COVERAGE':len(rated)/len(members),
        'ONET_WV_VERSION':'30.2', 'ONET_WV_SCALE':'EX',
        'ONET_WV_DATE_MIN':raw_ratings.rating_month.min().strftime('%Y-%m'),
        'ONET_WV_DATE_MAX':raw_ratings.rating_month.max().strftime('%Y-%m'),
        'ONET_WV_SOURCE':'; '.join(sorted(raw_ratings['Domain Source'].unique())),
        **selected.mean().to_dict()})
profiles = pd.DataFrame(profile_records)
candidate_audit = pd.DataFrame(candidate_records)
assert len(profiles) == 123 and profiles.ONET_PROFILE_ID.is_unique
assert not profiles[value_columns].isna().any().any()
assert not candidate_audit.duplicated(['N3OCPR', 'ONET_CODE']).any()
assert len(candidate_audit) == 858
# The CSV identifies group means without assigning an invented single SOC code.
assert profiles.loc[profiles.ONET_CANDIDATE_COUNT.gt(1), 'ONET_SOC_CODE'].eq('').all()
assert profiles.loc[profiles.ONET_CANDIDATE_COUNT.eq(1), 'ONET_SOC_CODE'].str.endswith('.00').all()
display(profiles.groupby(['MAPPING_TYPE', 'MAPPING_COVERAGE']).size().rename('Profiles').to_frame())
display(profiles[['ONET_PROFILE_ID', 'ONET_CANDIDATE_COUNT', 'ONET_RATED_COUNT', 'MAPPING_COVERAGE'] + value_columns].head())


,,Profiles
MAPPING_TYPE,MAPPING_COVERAGE,
group_mean_complete,full,24
group_mean_partial,partial,36
residual_single_proxy,full,1
single_SOC,full,57
single_SOC_broader_profile,full,3
single_available_proxy,partial,2


,ONET_PROFILE_ID,ONET_CANDIDATE_COUNT,ONET_RATED_COUNT,MAPPING_COVERAGE,ONET_WV_achievement,ONET_WV_working_conditions,ONET_WV_recognition,ONET_WV_relationships,ONET_WV_support,ONET_WV_independence
0,NSCG_110510,1,1,full,5.670,5.67,5.330,3.670,4.67,5.33
1,NSCG_110520,1,1,full,6.000,5.33,4.330,2.670,4.33,5.33
2,NSCG_110540,2,2,full,5.165,4.92,4.165,4.165,5.00,4.50
3,NSCG_110550,1,1,full,5.000,5.17,5.000,3.670,4.33,5.00
4,NSCG_110560,1,1,full,4.670,5.00,4.330,3.670,5.00,3.33


## 4. Left join and select respondents

Eligibility requires employment and a valid satisfaction response. Each left join preserves all eligible respondents. The supervisor screen and category decisions determine which rows receive a profile; excluded rows receive no Work Values.

Original `JOBSATIS` and `FAC*` codes remain 1–4, with lower values indicating greater satisfaction or importance. `WTSURVY` is unchanged.


In [5]:
satisfaction_labels = ['Very satisfied', 'Somewhat satisfied', 'Somewhat dissatisfied', 'Very dissatisfied']
valid_satisfaction = [code_for('JOBSATIS', label) for label in satisfaction_labels]
valid_preferences = {v:[c for c,label in labels_for(v).items() if label not in ['Logical Skip','Missing']] for v in fac}
assert all(set(codes)=={'1','2','3','4'} for codes in valid_preferences.values())
employed = survey.LFSTAT.eq(code_for('LFSTAT', 'Employed'))
eligible_mask = employed & survey.JOBSATIS.isin(valid_satisfaction)
eligible = survey.loc[eligible_mask].copy()
assert len(survey)==94606 and len(eligible)==79800
assert all(eligible[v].isin(valid_preferences[v]).all() for v in fac)
eligible['NSCG_OCC_TITLE'] = eligible.N3OCPR.map(published_titles)
assert eligible.NSCG_OCC_TITLE.notna().all()
bridge = rules[['code','decision','scope_note']].rename(columns={
    'code':'N3OCPR', 'decision':'CATEGORY_DECISION', 'scope_note':'MAPPING_SCOPE_NOTE'})
bridge['MAPPING_BASIS'] = 'Reviewed study occupation mapping'
linked = eligible.merge(bridge, on='N3OCPR', how='left', validate='many_to_one', indicator='bridge_join')
assert len(linked)==len(eligible) and linked.OBSNUM.is_unique
assert linked.bridge_join.eq('both').all()
assert linked.OBSNUM.tolist()==eligible.OBSNUM.tolist()
supervision_hold = linked.CATEGORY_DECISION.eq('include') & linked.N3OCPR.isin(['570980','780310']) & ~linked.SUPWK.eq(code_for('SUPWK','No'))
linked['INTEGRATION_STATUS'] = linked.CATEGORY_DECISION.replace({'include':'matched_reviewed'})
linked.loc[supervision_hold, 'INTEGRATION_STATUS'] = 'excluded_supervisor_screen'
matched = linked.INTEGRATION_STATUS.eq('matched_reviewed')
linked['ONET_PROFILE_ID'] = ('NSCG_' + linked.N3OCPR).where(matched)

integrated = linked.merge(profiles, on='ONET_PROFILE_ID', how='left', validate='many_to_one', indicator='profile_join')
matched = integrated.INTEGRATION_STATUS.eq('matched_reviewed')
assert len(integrated)==len(eligible) and integrated.OBSNUM.is_unique
assert integrated.OBSNUM.tolist()==eligible.OBSNUM.tolist()
assert integrated.profile_join.eq('both').equals(matched)
assert integrated.loc[~matched, value_columns].isna().all().all()
assert integrated.loc[matched, value_columns].notna().all().all()
assert integrated.INTEGRATION_STATUS.value_counts().to_dict()=={
    'matched_reviewed':74016, 'exclude_no_work_values':5068,
    'exclude_uncertain_mapping':548, 'excluded_supervisor_screen':168}
assert integrated.loc[matched].groupby('MAPPING_COVERAGE').size().to_dict()=={'full':39084,'partial':34932}
assert int(integrated.loc[matched, 'MAPPING_IS_PROXY'].sum())==23249
display(integrated.INTEGRATION_STATUS.value_counts().rename('People').to_frame())


,People
INTEGRATION_STATUS,
matched_reviewed,74016
exclude_no_work_values,5068
exclude_uncertain_mapping,548
excluded_supervisor_screen,168


## 5. Validate the exported subset

Check respondent IDs, every source field, each occupation count and all six values. The profile ID is an audit key, not a calibrated person-job fit measure.


In [6]:
derived = ['NSCG_OCC_TITLE','SOC2018_CODE','ONET_SOC_CODE','ONET_OCC_TITLE','ONET_PROFILE_ID',
    'MAPPING_BASIS','MAPPING_TYPE','MAPPING_SCOPE_NOTE','MAPPING_COVERAGE','MAPPING_IS_PROXY','MAPPING_PROXY_TYPE',
    'ONET_CANDIDATE_COUNT','ONET_RATED_COUNT','ONET_RATING_COVERAGE','INTEGRATION_STATUS',
    'ONET_WV_VERSION','ONET_WV_SCALE','ONET_WV_DATE_MIN','ONET_WV_DATE_MAX','ONET_WV_SOURCE'] + value_columns
subset = integrated.loc[matched, core+derived].sort_values('OBSNUM').reset_index(drop=True)
source_integer_fields = ['REFYR','AGE','JOBSATIS'] + fac
integer_fields = source_integer_fields + ['ONET_CANDIDATE_COUNT','ONET_RATED_COUNT']
for field in integer_fields:
    subset[field] = pd.to_numeric(subset[field], errors='raise').astype('Int64')
float_fields = ['WTSURVY','ONET_RATING_COVERAGE'] + value_columns
for field in float_fields:
    subset[field] = pd.to_numeric(subset[field], errors='raise').astype(float)
subset['MAPPING_IS_PROXY'] = subset.MAPPING_IS_PROXY.astype(bool)
for field in set(subset)-set(integer_fields)-set(float_fields)-{'MAPPING_IS_PROXY'}:
    subset[field] = subset[field].astype('string')
assert subset.shape==(74016,50) and subset.OBSNUM.is_unique
assert subset.N3OCPR.nunique()==subset.ONET_PROFILE_ID.nunique()==123
assert np.isfinite(subset.WTSURVY).all() and subset.WTSURVY.gt(0).all()
assert not subset.isna().any().any()
original = eligible.set_index('OBSNUM').loc[subset.OBSNUM, core[1:]].reset_index()
for field in core:
    if field in source_integer_fields+['WTSURVY']:
        np.testing.assert_array_equal(subset[field].to_numpy(float), pd.to_numeric(original[field]).to_numpy(float))
    else:
        assert subset[field].astype(str).tolist()==original[field].astype(str).tolist(), field
for field in value_columns:
    expected = subset.ONET_PROFILE_ID.map(profiles.set_index('ONET_PROFILE_ID')[field])
    np.testing.assert_array_equal(subset[field].to_numpy(), expected.to_numpy())
occupation_audit = rules[['code','title','sas_title','decision','eligible_n','keep_n','supervisor_excluded_n','scope_note']].copy()
occupation_audit['actual_eligible'] = occupation_audit.code.map(eligible.N3OCPR.value_counts()).fillna(0).astype(int)
occupation_audit['actual_included'] = occupation_audit.code.map(subset.N3OCPR.value_counts()).fillna(0).astype(int)
assert occupation_audit.actual_eligible.eq(occupation_audit.eligible_n).all()
assert occupation_audit.actual_included.eq(occupation_audit.keep_n).all()
assert occupation_audit.code.map(linked.loc[supervision_hold,'N3OCPR'].value_counts()).fillna(0).eq(occupation_audit.supervisor_excluded_n).all()
display(pd.Series({'Eligible rows after left joins':len(integrated), 'Included respondents':len(subset),
    'Duplicate respondent IDs':int(subset.OBSNUM.duplicated().sum()), 'NSCG categories':subset.N3OCPR.nunique(),
    'Columns':len(subset.columns)}, name='Result').to_frame())
display(subset[['OBSNUM','N3OCPR','ONET_PROFILE_ID','MAPPING_COVERAGE','MAPPING_IS_PROXY','JOBSATIS','FACIND','ONET_WV_independence']].head())
display(occupation_audit.loc[occupation_audit.decision.ne('include'), ['code','title','decision','actual_eligible']])


,Result
Eligible rows after left joins,79800
Included respondents,74016
Duplicate respondent IDs,0
NSCG categories,123
Columns,50


,OBSNUM,N3OCPR,ONET_PROFILE_ID,MAPPING_COVERAGE,MAPPING_IS_PROXY,JOBSATIS,FACIND,ONET_WV_independence
0,2023000002,782230,NSCG_782230,partial,False,2,1,3.856111
1,2023000003,621440,NSCG_621440,full,False,2,1,6.000000
2,2023000004,632530,NSCG_632530,full,True,1,1,5.000000
3,2023000005,640530,NSCG_640530,full,False,2,1,5.000000
4,2023000007,721510,NSCG_721510,partial,False,2,1,4.771818


,code,title,decision,actual_eligible
7,110590,Software developers – applications and systems software,exclude_no_work_values,4069
13,121760,Other mathematical scientists,exclude_no_work_values,79
14,121770,Data scientists,exclude_no_work_values,411
21,220270,Other life scientists,exclude_no_work_values,509
95,732570,Teachers: Other precollegiate area,exclude_uncertain_mapping,171
128,785000,Other occupations (not elsewhere classified),exclude_uncertain_mapping,377


## 6. Audit and column metadata

Save a respondent audit for all source rows, an occupation audit, candidate decisions, profile members, column definitions and response labels. These supporting files stay under ignored `data/private/`.


In [7]:
audit = survey[['OBSNUM','REFYR','LFSTAT','JOBSATIS','N3OCPR','N3OCPRX','SUPWK']].copy()
by_id = integrated.set_index('OBSNUM')
audit['INTEGRATION_STATUS'] = audit.OBSNUM.map(by_id.INTEGRATION_STATUS)
audit.loc[~employed,'INTEGRATION_STATUS'] = 'excluded_not_employed'
audit.loc[employed & ~eligible_mask,'INTEGRATION_STATUS'] = 'excluded_invalid_satisfaction'
audit['ONET_PROFILE_ID'] = audit.OBSNUM.map(by_id.ONET_PROFILE_ID)
assert len(audit)==94606 and audit.OBSNUM.is_unique and audit.INTEGRATION_STATUS.notna().all()
assert set(audit.loc[audit.INTEGRATION_STATUS.eq('matched_reviewed'),'OBSNUM'])==set(subset.OBSNUM)
profile_export = profiles.copy()
profile_export['N3OCPR'] = profile_export.ONET_PROFILE_ID.str.removeprefix('NSCG_')
spec_by_code = {r['code']:r for r in review_spec}
profile_export['CANDIDATE_CODES'] = profile_export.N3OCPR.map(lambda c:';'.join(spec_by_code[c]['candidates']))
profile_export['RATED_MEMBER_CODES'] = profile_export.N3OCPR.map(lambda c:';'.join(x for x in spec_by_code[c]['candidates'] if complete.at[x]))
derived_descriptions = {
 'NSCG_OCC_TITLE':'Published NSCG occupation title.',
 'SOC2018_CODE':'SOC2018 code only when the defined group contains one occupation; blank for groups.',
 'ONET_SOC_CODE':'Actual .00 code only for a one-occupation group; blank for multi-occupation groups, including single-rated-member proxies.',
 'ONET_OCC_TITLE':'Published title for one occupation; otherwise Group mean followed by the NSCG title.',
 'ONET_PROFILE_ID':'Study profile key NSCG_<N3OCPR>; not an official O*NET code.',
 'MAPPING_BASIS':'Study-specific reviewed mapping; not an official NSCG crosswalk.',
 'MAPPING_TYPE':'Single occupation, broader profile, group mean or single-member proxy.',
 'MAPPING_SCOPE_NOTE':'Reason for the membership and any scope limitations.',
 'MAPPING_COVERAGE':'full or partial coverage of the defined occupation members, not individual mapping accuracy.',
 'MAPPING_IS_PROXY':'True when an additional scope assumption or single-rated-member proxy is used.',
 'MAPPING_PROXY_TYPE':'Specific mapping assumption; none means no additional flagged assumption, not exact matching.',
 'ONET_CANDIDATE_COUNT':'Number of occupations in the defined member set.',
 'ONET_RATED_COUNT':'Members with six complete EX ratings; all six means use this same set.',
 'ONET_RATING_COVERAGE':'Rated member count divided by candidate count; not respondent coverage.',
 'INTEGRATION_STATUS':'Selection decision; only matched_reviewed rows enter the subset.',
 'ONET_WV_VERSION':'O*NET30.2.', 'ONET_WV_SCALE':'EX, original 1–7 scale.',
 'ONET_WV_DATE_MIN':'Earliest month among the ratings used in the profile (YYYY-MM).',
 'ONET_WV_DATE_MAX':'Latest month among the ratings used in the profile (YYYY-MM).',
 'ONET_WV_SOURCE':'Source methods among the underlying rating rows.',
}
dictionary_rows=[]
for field in subset:
    if field in core:
        role='identifier' if field=='OBSNUM' else 'outcome' if field=='JOBSATIS' else 'survey weight' if field=='WTSURVY' else 'personal preference' if field in fac else 'source covariate / audit'
        description=definitions.at[field,'DESCRIPTION']; source='NSCG 2023'; element=''; question=definitions.at[field,'QUESTION']
    elif field in value_columns:
        item=value_definitions.set_index('Output column').loc[field]
        role='occupational context'; description=item.Description; source='O*NET30.2 EX'; element=item['Element ID']; question=''
    else:
        role='mapping / provenance'; description=derived_descriptions[field]; source='Study mapping / source metadata'; element=''; question=''
    dictionary_rows.append({'column':field,'dtype':str(subset[field].dtype),'role':role,'source':source,
        'element_id':element,'description':description,'question':question})
column_dictionary=pd.DataFrame(dictionary_rows)
value_labels=pd.DataFrame([{'column':field,'code':c,'label':label} for field in core
    if pd.notna(definitions.at[field,'SAS_FMT']) for c,label in labels_for(field).items()])
assert column_dictionary.column.tolist()==subset.columns.tolist()
display(audit.INTEGRATION_STATUS.value_counts().rename('People').to_frame())


,People
INTEGRATION_STATUS,
matched_reviewed,74016
excluded_not_employed,14806
exclude_no_work_values,5068
exclude_uncertain_mapping,548
excluded_supervisor_screen,168


## 7. Save and reload

Save the integrated CSV beside this notebook and a second copy with the private audit files. Reload both with explicit dtypes and check exact equality. The manifest records the full mapping, source hashes, software versions and output hashes.


In [8]:
import subprocess
outputs = {'nscg2023_onet30_2_subset.csv':subset, 'respondent_audit.csv':audit,
    'occupation_audit.csv':occupation_audit, 'occupation_profiles.csv':profile_export,
    'candidate_audit.csv':candidate_audit, 'column_dictionary.csv':column_dictionary, 'value_labels.csv':value_labels}
for name in [*outputs, 'manifest.json']:
    result=subprocess.run(['git','check-ignore','--quiet',str(OUT/name)],cwd=ROOT)
    assert result.returncode==0, f'Expected data/private/ ignore rule: {name}'
OUT.mkdir(parents=True,exist_ok=True)
for name,frame in outputs.items():
    frame.to_csv(OUT/name,index=False,float_format='%.17g')
csv_dtypes=column_dictionary.set_index('column').dtype.to_dict()
reloaded=pd.read_csv(OUT/SUBSET_CSV.name,dtype=csv_dtypes,keep_default_na=False,float_precision='round_trip')
pd.testing.assert_frame_equal(subset,reloaded,check_exact=True)
subset.to_csv(SUBSET_CSV,index=False,float_format='%.17g')
notebook_csv=pd.read_csv(SUBSET_CSV,dtype=csv_dtypes,keep_default_na=False,float_precision='round_trip')
pd.testing.assert_frame_equal(subset,notebook_csv,check_exact=True)
assert SUBSET_CSV.read_bytes()==(OUT/SUBSET_CSV.name).read_bytes()
reloaded_audit=pd.read_csv(OUT/'respondent_audit.csv',dtype='string',keep_default_na=False)
assert len(reloaded_audit)==len(survey) and reloaded_audit.OBSNUM.is_unique
assert reloaded_audit.INTEGRATION_STATUS.value_counts().to_dict()==audit.INTEGRATION_STATUS.value_counts().to_dict()
def sha256(path):
    digest=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):
            digest.update(chunk)
    return digest.hexdigest()
manifest={
 'pipeline_version':'nscg2023_onet30_2_v2_2026-10-31', 'notebook':str(NOTEBOOK_REL),
 'grain':'One NSCG2023 respondent per row, identified by OBSNUM.',
 'concordance':{'source':'Embedded reviewed member lists in Section 2',
    'sha256':hashlib.sha256(json.dumps(review_spec,sort_keys=True).encode()).hexdigest(), 'spec':review_spec},
 'rules':{'eligibility':'LFSTAT=1 and JOBSATIS in 1,2,3,4', 'occupation_key':'N3OCPR',
    'supervision':'SUPWK=N for 570980 and 780310',
    'ratings':'Actual .00 profiles only; same fully rated members for all six equal occupation means',
    'merge':'Two many_to_one left joins: N3OCPR to category rules, screened ONET_PROFILE_ID to profiles; export included subset',
    'flags':'Partial coverage and proxy assumptions are separate; shared and single-rated-member profiles allowed',
    'exclusions':'Four categories without Work Values; 732570 and 785000 for uncertain mapping',
    'source_fields':'Preserve JOBSATIS, nine FAC items and WTSURVY; no fit score or model'},
 'counts':{'source':len(survey),'eligible':len(eligible),'included':len(subset),
    'included_categories':123,'excluded_categories':6,'no_work_values':5068,'uncertain_mapping':548,
    'supervisor_screen':168,'full_coverage':39084,'partial_coverage':34932,'proxy':23249},
 'sources':{key:{'path':str(p.relative_to(ROOT)),'sha256':sha256(p)} for key,p in SOURCE.items()},
 'outputs':{name:{'path':str((OUT/name).relative_to(ROOT)),'rows':len(frame),'columns':len(frame.columns),
    'sha256':sha256(OUT/name)} for name,frame in outputs.items()},
 'notebook_subset_csv':{'path':str(SUBSET_CSV.relative_to(ROOT)),'rows':len(subset),
    'columns':len(subset.columns),'sha256':sha256(SUBSET_CSV)},
 'software':{'python':platform.python_version(),'pandas':pd.__version__,'numpy':np.__version__},
 'limitations':['Study-defined mapping, not verified individual duties.',
    'Partial groups omit unrated members; group means use equal occupation weights.',
    'Broad profiles may include non-teaching or other duties outside the NSCG label.',
    'Shared occupation ratings are not independent workplace measurements.',
    'Survey completeness may include imputations; personal and occupational scales are not calibrated.'],
}
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
display(pd.DataFrame([{'File':name,'Rows':len(frame),'Columns':len(frame.columns)} for name,frame in outputs.items()]))
print(f'Integrated CSV: {SUBSET_CSV.relative_to(ROOT)}')
print('Both CSV copies reloaded exactly. All 79,800 eligible rows preserved through the left joins.')


Integrated CSV: notebooks/possible data integration/nscg2023_onet30_2_subset.csv
Both CSV copies reloaded exactly. All 79,800 eligible rows preserved through the left joins.


,File,Rows,Columns
0,nscg2023_onet30_2_subset.csv,74016,50
1,respondent_audit.csv,94606,9
2,occupation_audit.csv,129,10
3,occupation_profiles.csv,123,25
4,candidate_audit.csv,858,14
5,column_dictionary.csv,50,7
6,value_labels.csv,417,3


## Output and limits

The [50-column CSV](nscg2023_onet30_2_subset.csv) contains 24 NSCG fields, six Work Values and 20 mapping/provenance fields. Group means have a study profile ID and blank single-SOC fields; member lists are in the embedded rules and private profile export.

These are occupational backgrounds, not measurements of each workplace. Partial groups omit unrated members; proxy assumptions remain visible. Personal importance and occupational values have no calibrated common scale. Later evaluation should account for shared and overlapping occupation profiles.
